# Material de estudio personal: Annealing Parte 2 - Simulated Annealing

In [1]:
!pip install OceanSDK

"pip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


 ## Modelos de temperatura y recocido térmico (Annealing Físico)

En un problema de optimización combinatoria que hayamos representado utilizando una formulación QUBO o Ising en donde busquemos una energía mínima o máxima, podriamos pensar "Ok, que tal si tomo una solución aleatoria, y cambio uno de los bits de mi solución. Si mejora, tomo la solución mejorada, si no lo hace, regreso a mi solución y pruebo otro bit, y asi voy optimizando hasta que ya no pueda conseguir una solución mejor"

Es una buena estrategia pero tiene un problema: Los mínimos locales. Por ejemplo, puede ser que en una función QUBO, la solución $\bar{x}_1=11001$ ya no mejore si cambias un bit. Pero si cambias 3 bits, a $\bar{x}_2=10111$, esta SI MEJORA con respecto a la anterior. El valor $\bar{x}_1$ es un mínimo local, que si es un mínimo, pero solo en su vecindad, mientras que otros valores pueden ser menores, pero ser alejados a este

¿Como tajamos este problema?

Aquí entra el metodo Simulated Annealing. Este método esta inspirado en el "Annealing" físico, en el cual un material muy caliente puede explorar simultaneamente diversos valores de energía y distintas configuraciones posibles. Pero al bajar la temperatura, se va definiendo lentamente. Este principio se inspira detrás de los principios de la física estadística y las distribuciones de Boltzmann que lo justifican.

## Simulated Annealing: Principios de Selección y Procesos Físicos

### Distribuciones de Boltzmann y Física Estadística

Si eres un físico, seguramente recordaras el ensamble canónico. Este ensamble nos dice que si un sistema se encuentra a una temperatura T, la probabilidad de encontrarlo en un estado x con energía $E(x)$ es de:

$$P_T(x)=\frac{e^{-E(x)/T}}{Z(T)}$$

Donde $Z(T)=\sum_xe^{-E(x)/T}$

Nota como si T es menor, la probabilidad se vuelve exponencialmente mayor. Ese asunto nos dice algo interesante que podremos usar despues como inspiración. Veamos como esto puede evolucionar si tomamos la razón de 2 probabilidades

$$\frac{P_T(y)}{P_T(x)}=\frac{e^{-E(y)/T}}{e^{-E(x)/T}}=e^{-\Delta E/T}$$

Si $y$ tiene menor energía que $x$, entonces $\Delta E < 0$ y $e^{-\Delta E/T}> 1$, es decir, $y$ es mas probable que $x$

Ahora, si T es muy alta, las diferencias de energía pesan poco, porque la exponencial tiende a 1, pero si las temperaturas son bajas, las diferencias de energía pesan bastante en la exponencial. Todo esto nos sugiere que podemos controlar el parámetro para controlar las probabilidades de que algo surga.

### Método de selección de metropolis

Imagina que tiene una configuración detras de la cual partes. La conf $x$. Y propones una configuración diferente, $x'$. Entonces calculas $\Delta E= E(x')-E(x)$. ¿Cuando tomas y cuando lo dejas?

Tienes 2 posibilidades, si $\Delta E$ es menor o igual que 0, o si es mayor que 0

Si $\Delta E \leq 0$ siempre se acepta, pues es un valor mejor

El problema surge si $\Delta E \geq 0$

En Simulated Annealing, aceptas este caso con probabilidad $e^{-\Delta E/T}$ donde T es un parámetro que tu decides controlar. Depende de 2 cosas, de la temperatura y de la diferencia de energía. Esto permite que brinques entre casos que tengan energías similares y la temperatura T se vuelve un parámetro moderado.



### Vecindad y selección de vecinos

El simulated Annealing se mueve a través de Vecinos. ¿Como se realiza esto?

Lo que haces es que realizas el cambio de un solo bit, y al hacerlo, este se le llama vecino del otro bit. En el caso de QUBO, un Vecino es aquel que cambia un valor de 1 a 0, o de 0 a 1. Y en el caso de Ising, de 1 a -1 o de -1 a 1

### Cooling schedule

Tu vas a controlar la temperatura del sistema y como esta evoluciona lentamente con el tiempo. Tienes que definir una secuencia de valores de temperatura $T_0>T_1>...>T_f$. La evolución de temperatura que asignes depende de ti. Los enfoques de evolución de temperatura son varios:

- **Geométrico**: Asignas un valor a $\alpha$ para iterar las temperaturas

$$ T_{k+1}=\alpha T_k $$

- **Lineal**: Evolucionas a forma de recta

$$T_k =T_0 - k\Delta T$$

- **Logaritmico**: Evoluciona mas lentamente al final, pero resulta muy lento

$$T_k=\frac{C}{\log(k+1)}$$

## Simulación Computacional

Utilizaremos el simulador de D-Wave para correr el Simulated Annealing para un BQM. Usaremos el mismo BQM que fabricamos en la sesión anterior

In [6]:
from dimod import BQM
from dwave.samplers import SimulatedAnnealingSampler


sampler = SimulatedAnnealingSampler()  #Usaremos un sampler de Simulated Annealing dado por D-Wave 

linear = {"x1": -3, "x2": -5, "x3": -7, "x4": -4}
quadratic = {("x1","x2"): 3,("x2","x3"): 4, ("x3","x4"): 5, ("x1","x4"): 2}
vartype = 'BINARY'

bqm = BQM(linear, quadratic, vartype)

sampleset = sampler.sample(bqm, num_reads=10) #Realizamos 10 lecturas para obtener diferentes soluciones al problema planteado
print(sampleset)

  x1 x2 x3 x4 energy num_oc.
1  1  0  1  0  -10.0       1
3  1  0  1  0  -10.0       1
4  1  0  1  0  -10.0       1
6  1  0  1  0  -10.0       1
7  1  0  1  0  -10.0       1
8  1  0  1  0  -10.0       1
9  1  0  1  0  -10.0       1
0  0  1  0  1   -9.0       1
2  0  1  0  1   -9.0       1
5  0  1  0  1   -9.0       1
['BINARY', 10 rows, 10 samples, 4 variables]


## Referencias

La información técnica sobre formulación QUBO/Ising/BQM, uso de Ocean SDK y samplers de D-Wave fue tomada o contrastada con los materiales de **QCobalt/QWorld** y con la documentación oficial de **D-Wave Ocean SDK**.  
La explicación teórica sobre ensamble canónico, distribución de Boltzmann, criterio de Metrópolis, vecindades y recocido simulado fue redactada por el autor a partir de conocimiento propio de física estadística y optimización; se incluyen referencias canónicas para trazabilidad académica.

### Material técnico y documentación

1. QWorld. *QCobalt: QWorld’s intermediate level workshop series on quantum annealing*. QWorld.  
   Disponible en: https://qworld.net/qcobalt/

2. QWorld. *Cobalt: Educational material for quantum annealing*. GitLab repository.  
   Disponible en: https://gitlab.com/qworld/cobalt

3. D-Wave Systems Inc. *Ocean SDK Documentation*.  
   Disponible en: https://docs.dwavequantum.com/en/latest/

4. D-Wave Systems Inc. *dimod: Binary Quadratic Models*. Ocean SDK Documentation.  
   Disponible en: https://docs.dwavequantum.com/en/latest/ocean/api_ref_dimod/models.html

5. D-Wave Systems Inc. *dwave.samplers.SimulatedAnnealingSampler.sample*. Ocean SDK Documentation.  
   Disponible en: https://docs.dwavequantum.com/en/latest/ocean/api_ref_samplers/generated/dwave.samplers.SimulatedAnnealingSampler.sample.html

6. D-Wave Systems Inc. *dwave-samplers: Classical samplers for binary quadratic models*. Ocean SDK Documentation.  
   Disponible en: https://docs.dwavequantum.com/en/latest/ocean/api_ref_samplers/index.html

### Referencias teóricas

7. Metropolis, N., Rosenbluth, A. W., Rosenbluth, M. N., Teller, A. H., & Teller, E. (1953).  
   *Equation of State Calculations by Fast Computing Machines*.  
   The Journal of Chemical Physics, 21(6), 1087–1092.  
   https://doi.org/10.1063/1.1699114

8. Kirkpatrick, S., Gelatt, C. D., & Vecchi, M. P. (1983).  
   *Optimization by Simulated Annealing*.  
   Science, 220(4598), 671–680.  
   https://doi.org/10.1126/science.220.4598.671

9. Černý, V. (1985).  
   *Thermodynamical Approach to the Traveling Salesman Problem: An Efficient Simulation Algorithm*.  
   Journal of Optimization Theory and Applications, 45, 41–51.  
   https://doi.org/10.1007/BF00940812

10. Geman, S., & Geman, D. (1984).  
    *Stochastic Relaxation, Gibbs Distributions, and the Bayesian Restoration of Images*.  
    IEEE Transactions on Pattern Analysis and Machine Intelligence, PAMI-6(6), 721–741.  
    https://doi.org/10.1109/TPAMI.1984.4767596

11. Pathria, R. K., & Beale, P. D. (2011).  
    *Statistical Mechanics* (3rd ed.). Elsevier.

12. Landau, D. P., & Binder, K. (2021).  
    *A Guide to Monte Carlo Simulations in Statistical Physics* (5th ed.). Cambridge University Press.

### Nota de autoría

Las derivaciones, explicaciones intuitivas, ejemplos narrativos y conexión física entre distribución de Boltzmann, criterio de Metrópolis y recocido simulado fueron elaboradas por el autor como material de estudio personal. Las fuentes anteriores se citan para respaldar los conceptos estándar y la implementación computacional utilizada.